# Model 01: spread of genealogical founder ancestry\n\nThis notebook explores a deliberately simple baseline model. It tracks **genealogical ancestry**, not DNA.\n\nChange the controls below and rerun the simulation. The important question is not whether this baseline is historically realistic. The question is how its behavior changes once assumptions are made explicit.

In [ ]:
import os, sys, subprocess\nif 'google.colab' in sys.modules:\n    if not os.path.exists('/content/Evolution-Creation'):\n        subprocess.run(['git', 'clone', '-q', 'https://github.com/vafaei-ar/Evolution-Creation.git', '/content/Evolution-Creation'], check=True)\n    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/Evolution-Creation'], check=True)\n

In [ ]:
import matplotlib.pyplot as plt\nimport numpy as np\nimport ipywidgets as widgets\nfrom IPython.display import display\nfrom evolution_creation.genealogy import deterministic_random_mating_curve, simulate_replicates\n

In [ ]:
population = widgets.IntSlider(value=1000, min=50, max=10000, step=50, description='N')\nfounders = widgets.IntSlider(value=1, min=1, max=20, step=1, description='Founders')\ngenerations = widgets.IntSlider(value=25, min=1, max=100, step=1, description='Generations')\nreplicates = widgets.IntSlider(value=200, min=10, max=1000, step=10, description='Replicates')\nseed = widgets.IntText(value=20260920, description='Seed')\ndisplay(population, founders, generations, replicates, seed)

In [ ]:
def run_model(_=None):\n    curves = simulate_replicates(\n        population_size=population.value,\n        generations=generations.value,\n        founder_count=min(founders.value, population.value),\n        replicates=replicates.value,\n        seed=seed.value,\n    )\n    x = np.arange(generations.value + 1)\n    median = np.median(curves, axis=0)\n    lo = np.quantile(curves, 0.10, axis=0)\n    hi = np.quantile(curves, 0.90, axis=0)\n    deterministic = deterministic_random_mating_curve(founders.value / population.value, generations.value)\n\n    fig, ax = plt.subplots(figsize=(9, 5))\n    ax.fill_between(x, lo, hi, alpha=0.2, label='10th-90th percentile')\n    ax.plot(x, median, label='simulation median')\n    ax.plot(x, deterministic, '--', label='deterministic approximation')\n    ax.set(xlabel='Generation', ylabel='Fraction with founder ancestry', ylim=(0, 1.02))\n    ax.legend()\n    plt.show()\n\n    print(f'Lineage extinct by final generation: {(curves[:, -1] == 0).mean():.1%}')\n    print(f'Founder ancestry fixed by final generation: {(curves[:, -1] == 1).mean():.1%}')\n\nrun_button = widgets.Button(description='Run simulation', button_style='primary')\nrun_button.on_click(run_model)\ndisplay(run_button)\nrun_model()

## Interpretation\n\nIf founder ancestry spreads rapidly here, that result applies only to this completely mixed baseline. The next models will add population structure and migration barriers. Those additions are essential for evaluating claims about real human history.